In [1]:
import json
from datasets import load_dataset
import pandas as pd
import sqlglot
from sqlglot import exp
import re

d:\thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_PATH = 'xlangai/spider'

In [3]:
dataset = load_dataset(DATASET_PATH)

In [4]:
statements = list(dataset['train']['query']) + list(dataset['validation']['query'])

In [5]:
# remove unnecessary whitespace and newlines
for statement in statements:
    statement = statement.strip()

In [6]:
print(statements[0])

SELECT count(*) FROM head WHERE age  >  56


In [7]:
index = 0
keywords = []

In [8]:
for index, statement in enumerate(statements):
    try:
        # Quan trọng: Nên chuẩn hóa khoảng trắng TRƯỚC khi parse để tránh lỗi ký tự lạ
        statement = re.sub(r'\s+', ' ', statement).strip()
        parsed = sqlglot.parse_one(statement)

        features = {
            "tables": [t.name.lower() for t in parsed.find_all(exp.Table)],
            
            # TRÍCH XUẤT CỘT TRONG SELECT
            "columns_select": [
                c.name.lower() 
                for c in parsed.find(exp.Select).find_all(exp.Column)
            ] if parsed.find(exp.Select) else [],

            # TRÍCH XUẤT CỘT TRONG JOIN ON
            "join_columns": [
                c.name.lower() 
                for j in parsed.find_all(exp.Join) 
                if j.args.get("on") 
                for c in j.args["on"].find_all(exp.Column)
            ],

            "columns_where": [c.name.lower() for c in parsed.find(exp.Where).find_all(exp.Column)] if parsed.find(exp.Where) else [],
            
            "columns_group": [c.name.lower() for c in parsed.find(exp.Group).find_all(exp.Column)] if parsed.find(exp.Group) else [],

            # TRÍCH XUẤT CỘT TRONG ORDER BY
            "columns_order": [
                c.name.lower() 
                for c in parsed.find(exp.Order).find_all(exp.Column)
            ] if parsed.find(exp.Order) else [],

            "join_types": [j.args.get("side") for j in parsed.find_all(exp.Join) if j.args.get("side")],
            "timestamp": index
        }
        keywords.append(features)

    except Exception as e:
        print(f"Error parsing statement at index {index}: {statement}")
        print(f"Detail: {e}")
        continue
    

In [9]:
print(keywords[0:5])

[{'tables': ['head'], 'columns_select': ['age'], 'join_columns': [], 'columns_where': ['age'], 'columns_group': [], 'columns_order': [], 'join_types': [], 'timestamp': 0}, {'tables': ['head'], 'columns_select': ['name', 'born_state', 'age', 'age'], 'join_columns': [], 'columns_where': [], 'columns_group': [], 'columns_order': ['age'], 'join_types': [], 'timestamp': 1}, {'tables': ['department'], 'columns_select': ['creation', 'name', 'budget_in_billions'], 'join_columns': [], 'columns_where': [], 'columns_group': [], 'columns_order': [], 'join_types': [], 'timestamp': 2}, {'tables': ['department'], 'columns_select': ['budget_in_billions', 'budget_in_billions'], 'join_columns': [], 'columns_where': [], 'columns_group': [], 'columns_order': [], 'join_types': [], 'timestamp': 3}, {'tables': ['department'], 'columns_select': ['num_employees', 'ranking'], 'join_columns': [], 'columns_where': ['ranking'], 'columns_group': [], 'columns_order': [], 'join_types': [], 'timestamp': 4}]


In [11]:
with open('../preprocessed_data/spider_statements.json', 'w') as f:
    json.dump(keywords, f, indent=4)